# NHL-Beyond-27 · Book 3 — Spicy & Spicy-Weighted Analysis

Repo: `ewnike/NHL-Beyond-27` — *MADS Milestone I Project*  
Python: 3.13.7 (pyenv env: `nhl_beyond27-3.13.7`)  
Editors/Tools: VSCode, Git/GitHub, Postgres + pgAdmin  
Logs: written via `log_utils.py` (default `logs/`)

**This notebook covers:**
1) Load analysis view/table  
2) Define roles (Defense vs Forwards)  
3) Averages of **spicy** and **spicy_weighted** by role over `rel_age ∈ {-2,-1,0,1,2}`  
4) Visuals: lines with error bars, histograms, and scatter (spicy vs spicy_weighted)  
5) Optional: per-player 5-year averages table for quick reference


In [8]:
from pathlib import Path

import pandas as pd

USE_DB = True  # set False to force CSV path

df = None
err = None

if USE_DB:
    try:
        from db_utils import get_db_engine
        engine = get_db_engine()
        # View columns expected:
        # player, position, peak_year, rel_age, season, age,
        # cf_pct_z, cf60_z, ca60_z, spicy_score, spicy_weighted,
        # cf_pct_dz, cf60_dz, ca60_dz, spicy_unw_dz, spicy_w_dz
        df = pd.read_sql("SELECT * FROM public.v_player_spicy_by_rel_age", engine)
        print(f"Loaded {len(df):,} rows from DB view.")
    except Exception as e:
        err = e
        print("DB load failed; will try CSV fallback. Details:", e)

if df is None:
    # Fallback CSV path — adjust if you exported under a different name
    csv_fallback = Path("data/outputs/player_five_year_aligned_z.csv")
    assert csv_fallback.exists(), (
        "CSV fallback not found. Export the z-table to CSV or enable DB connection."
    )
    df = pd.read_csv(csv_fallback)
    print(f"Loaded {len(df):,} rows from CSV fallback:", csv_fallback)

df.head(5)


2025-09-28 22:57:04,669 - INFO - db_utils - Using DATABASE_URL from environment.


DB load failed; will try CSV fallback. Details: (psycopg2.errors.UndefinedTable) relation "public.v_player_spicy_by_rel_age" does not exist
LINE 1: SELECT * FROM public.v_player_spicy_by_rel_age
                      ^

[SQL: SELECT * FROM public.v_player_spicy_by_rel_age]
(Background on this error at: https://sqlalche.me/e/20/f405)


AssertionError: CSV fallback not found. Export the z-table to CSV or enable DB connection.

In [7]:
# Verify imports

import numpy as np
import pandas as pd
import plotly.express as px
import statsmodels.api as sm
import statsmodels.formula.api as smf

print("numpy", np.__version__)
print("pandas", pd.__version__)
print("plotly", px.__version__)
print("statsmodels", sm.__version__)


numpy 2.3.2
pandas 2.3.2


AttributeError: module 'plotly.express' has no attribute '__version__'

In [ ]:
import numpy as np


def to_role(pos: str) -> str:
    s = str(pos or "").strip().upper()
    return "D" if s.startswith("D") else "F"  # F = forwards / everyone else

req_cols = {"position", "rel_age", "spicy_score", "spicy_weighted"}
missing = req_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

df["role"] = df["position"].map(to_role)
df = df[df["rel_age"].isin([-2, -1, 0, 1, 2])].copy()
df = df.replace([np.inf, -np.inf], np.nan)

print(df["role"].value_counts(dropna=False))
df.head(3)


In [ ]:
def agg_ci(g, col):
    s = pd.to_numeric(g[col], errors="coerce")
    n = s.notna().sum()
    mean = s.mean()
    std = s.std(ddof=1)
    se = std / np.sqrt(n) if n > 0 else np.nan
    ci95 = 1.96 * se if n > 0 else np.nan
    return pd.Series({"n": n, "mean": mean, "std": std, "se": se, "ci95": ci95})

agg_spicy = df.groupby(["role", "rel_age"]).apply(agg_ci, "spicy_score").reset_index()
agg_spicy_w = df.groupby(["role", "rel_age"]).apply(agg_ci, "spicy_weighted").reset_index()

display(agg_spicy.head(10))
display(agg_spicy_w.head(10))


In [ ]:
import plotly.express as px

fig1 = px.line(
    agg_spicy,
    x="rel_age",
    y="mean",
    color="role",
    error_y="ci95",
    markers=True,
    title="Mean Spicy (±95% CI) by rel_age and Role",
    labels={"mean": "spicy (z)", "rel_age": "rel_age", "role": "Role"},
)
fig1.update_layout(yaxis=dict(zeroline=True))  # noqa: C408
fig1.show()

fig2 = px.line(
    agg_spicy_w,
    x="rel_age",
    y="mean",
    color="role",
    error_y="ci95",
    markers=True,
    title="Mean Spicy-Weighted (±95% CI) by rel_age and Role",
    labels={"mean": "spicy_weighted (z)", "rel_age": "rel_age", "role": "Role"},
)
fig2.update_layout(yaxis=dict(zeroline=True))  # noqa: C408
fig2.show()


In [ ]:
fig_h = px.histogram(
    df,
    x="spicy_weighted",
    color="role",
    barmode="overlay",
    nbins=40,
    title="Distribution of spicy_weighted by Role (all rel_age pooled)",
    labels={"spicy_weighted": "spicy_weighted (z)"},
    opacity=0.65,
)
fig_h.update_layout(bargap=0.02)
fig_h.show()


In [ ]:
fig_sc = px.scatter(
    df,
    x="spicy_score",
    y="spicy_weighted",
    color="role",
    symbol="role",
    facet_col="rel_age",
    facet_col_wrap=5,
    trendline="ols",  # if statsmodels is available; otherwise remove this line
    title="Spicy vs Spicy-Weighted by Role (faceted by rel_age)",
    labels={"spicy_score": "spicy", "spicy_weighted": "spicy_weighted"},
    opacity=0.6,
)
fig_sc.update_layout(showlegend=True)
fig_sc.show()


In [ ]:
summary = (
    df.groupby(["role", "rel_age"])
      .agg(
          spicy_mean=("spicy_score", "mean"),
          spicy_w_mean=("spicy_weighted", "mean"),
          n=("spicy_score", "count"),
      )
      .reset_index()
      .sort_values(["role", "rel_age"])
)
summary.round({"spicy_mean": 3, "spicy_w_mean": 3}).head(20)


In [ ]:
player_5yr = (
    df.groupby(["player", "role"])
      .agg(
          spicy_mean=("spicy_score", "mean"),
          spicy_std=("spicy_score", "std"),
          spicy_w_mean=("spicy_weighted", "mean"),
          spicy_w_std=("spicy_weighted", "std"),
          years=("rel_age", "nunique"),
      )
      .reset_index()
)
player_5yr = player_5yr[player_5yr["years"].ge(3)]  # keep players with ≥3 seasons in window (optional)
player_5yr.round(3).head(20)


In [ ]:
OUT = Path("data/outputs")
OUT.mkdir(parents=True, exist_ok=True)

summary.to_csv(OUT / "spicy_summary_by_role_rel_age.csv", index=False)
player_5yr.to_csv(OUT / "spicy_player_5yr_summary.csv", index=False)

print("Wrote:",
      OUT / "spicy_summary_by_role_rel_age.csv",
      "and",
      OUT / "spicy_player_5yr_summary.csv")


## Notes & interpretation

- **Within-player standardization**: both spicy and spicy_weighted are in z-score units relative to each player’s own 5-year baseline.
- **Role split**: we classify `position` starting with `D` as Defense, everything else as Forwards (C/LW/RW…).
- **rel_age axis**: `-2,-1,0,1,2` aligns to ages like 25–29 in our cohort build (peak = 0).
- **What to look for**:
  - Do Defense (“D”) show flatter/less volatile spicy_weighted curves than Forwards?
  - Does the weighting visibly nudge defensemen down (penalizing CA/60) and forwards up (rewarding CF/60)?
  - Are histograms tighter/wider by role?


In [ ]:
wide = summary.pivot(index="rel_age", columns="role", values="spicy_w_mean")
wide["F_minus_D"] = wide.get("F", pd.Series(index=wide.index)) - wide.get("D", pd.Series(index=wide.index))
wide.round(3)


In [ ]:
import numpy as np
import pandas as pd

# statsmodels for OLS
import statsmodels.formula.api as smf

# We’ll reuse df from earlier cells. Ensure it has the columns we need and no infs/NAs in the target.
need = {"player", "role", "rel_age", "spicy_score", "spicy_weighted"}
missing = need - set(df.columns)
if missing:
    raise ValueError(f"Missing columns for regression: {sorted(missing)}")

reg = df.copy()
reg = reg.replace([np.inf, -np.inf], np.nan)

# Keep the rel_age window we care about and drop rows missing the target
reg = reg[reg["rel_age"].isin([-2,-1,0,1,2])].copy()

# Role: ensure exactly {"D","F"} with "D" as reference in the model
def to_role(pos):
    s = str(pos or "").strip().upper()
    return "D" if s.startswith("D") else "F"

reg["role"] = reg["role"].map(lambda x: "D" if str(x).upper().startswith("D") else "F")

# Drop rows without outcome
reg_sw = reg.dropna(subset=["spicy_weighted"]).copy()
reg_s  = reg.dropna(subset=["spicy_score"]).copy()

print(f"Rows for spicy_weighted: {len(reg_sw):,} | Rows for spicy: {len(reg_s):,}")
reg_sw[["player","role","rel_age","spicy_weighted"]].head(3)


In [ ]:
# C(role, Treatment('D')) makes "D" the reference; C(rel_age, Treatment(0)) makes 0 the reference
fml_sw = "spicy_weighted ~ C(role, Treatment('D')) * C(rel_age, Treatment(0))"

model_sw = smf.ols(formula=fml_sw, data=reg_sw).fit(cov_type="HC3")
print(model_sw.summary())


In [ ]:
fml_s = "spicy_score ~ C(role, Treatment('D')) * C(rel_age, Treatment(0))"

model_s = smf.ols(formula=fml_s, data=reg_s).fit(cov_type="HC3")
print(model_s.summary())


In [ ]:
# Cluster-robust by player; useful if multiple rows per player (which we have)
# Note: statsmodels accepts group labels as strings.
model_sw_cl = smf.ols(formula=fml_sw, data=reg_sw).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_sw["player"]},
)
print(model_sw_cl.summary())


In [ ]:
import plotly.express as px

# Grid of unique combos to predict
grid = (
    pd.MultiIndex.from_product([["D","F"], [-2,-1,0,1,2]], names=["role","rel_age"])
    .to_frame(index=False)
)

# Predict with CI from the HC3 model
pred = model_sw.get_prediction(grid).summary_frame(alpha=0.05)
pred = pd.concat([grid.reset_index(drop=True), pred.reset_index(drop=True)], axis=1)
pred.rename(columns={"mean":"yhat","mean_ci_lower":"ci_lo","mean_ci_upper":"ci_hi"}, inplace=True)

pred.head(10)


In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

for role, g in pred.groupby("role"):
    g = g.sort_values("rel_age")
    # CI ribbon
    fig.add_traces([
        go.Scatter(
            x=g["rel_age"], y=g["ci_hi"],
            mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip"
        ),
        go.Scatter(
            x=g["rel_age"], y=g["ci_lo"],
            mode="lines", line=dict(width=0), fill="tonexty",
            name=f"{role} 95% CI", hoverinfo="skip", opacity=0.2
        ),
    ])
    # Mean line
    fig.add_trace(
        go.Scatter(
            x=g["rel_age"], y=g["yhat"],
            mode="lines+markers",
            name=f"{role} mean",
        )
    )

fig.update_layout(
    title="Predicted spicy_weighted by rel_age × role (OLS, HC3 CIs)",
    xaxis_title="rel_age",
    yaxis_title="spicy_weighted (z)",
    legend_title="Series",
)
fig.show()


In [ ]:
# Pull coefficients into a tidy table. In this parameterization,
# Intercept = D @ rel_age=0
coefs = (
    model_sw.summary2().tables[1]
    .rename_axis("term")
    .reset_index()
    .rename(columns={"Coef.":"coef","Std.Err.":"se","P>|t|":"p"})
)[["term","coef","se","p"]]

# Helpful labels
coefs["label"] = coefs["term"].replace({
    "Intercept": "D @ rel_age=0",
    "C(role, Treatment('D'))[T.F]": "F vs D (at rel_age=0)",
    "C(rel_age, Treatment(0))[T.-2]": "rel_age -2 vs 0 (Defense)",
    "C(rel_age, Treatment(0))[T.-1]": "rel_age -1 vs 0 (Defense)",
    "C(rel_age, Treatment(0))[T.1]":  "rel_age 1 vs 0 (Defense)",
    "C(rel_age, Treatment(0))[T.2]":  "rel_age 2 vs 0 (Defense)",
    "C(role, Treatment('D'))[T.F]:C(rel_age, Treatment(0))[T.-2]": "Interaction F×(-2)",
    "C(role, Treatment('D'))[T.F]:C(rel_age, Treatment(0))[T.-1]": "Interaction F×(-1)",
    "C(role, Treatment('D'))[T.F]:C(rel_age, Treatment(0))[T.1]":  "Interaction F×1",
    "C(role, Treatment('D'))[T.F]:C(rel_age, Treatment(0))[T.2]":  "Interaction F×2",
})

coefs.round({"coef":3,"se":3,"p":4}).sort_values("term").head(20)


## Regression interpretation (quick)

- **Reference point** is Defense at `rel_age=0` (the Intercept).
- A positive **F vs D** coefficient means Forwards exceed Defense **at rel_age=0**.
- **rel_age k vs 0 (Defense)** coefficients show how Defense deviates from their peak (`0`) at each k ∈ {-2,-1,1,2}.
- **Interaction terms** `F×k` indicate how the Forward–Defense gap changes at each rel_age k relative to rel_age 0.
- The **prediction plot** shows adjusted group means and 95% CIs from the OLS with HC3 robust errors.

*If clustered SEs (by player) are preferred, use the `model_sw_cl` results; coefficients are the same, but SEs and p-values reflect within-player dependence.*


In [ ]:
reg_sw_lin = reg_sw.copy()
# Ensure rel_age is numeric
reg_sw_lin["rel_age"] = reg_sw_lin["rel_age"].astype(int)

fml_sw_lin = "spicy_weighted ~ C(role, Treatment('D')) + rel_age + C(role, Treatment('D')):rel_age"
model_sw_lin = smf.ols(formula=fml_sw_lin, data=reg_sw_lin).fit(cov_type="HC3")
print(model_sw_lin.summary())


In [ ]:
from pathlib import Path

OUT = Path("data/outputs")
OUT.mkdir(parents=True, exist_ok=True)

# 1) Predictions by role × rel_age (from HC3 model)
pred_out = pred.copy()
pred_out.to_csv(OUT / "reg_spicy_weighted_preds_by_role_rel_age.csv", index=False)

# 2) Coefficients / SEs / p-values (HC3)
effects_out = coefs.copy()
effects_out.to_csv(OUT / "reg_spicy_weighted_effects_hc3.csv", index=False)

# 3) Optional: clustered SEs by player (same coefficients, different SE/p)
try:
    coefs_cl = (
        model_sw_cl.summary2().tables[1]
        .rename_axis("term")
        .reset_index()
        .rename(columns={"Coef.":"coef","Std.Err.":"se","P>|t|":"p"})
    )[["term","coef","se","p"]]
    coefs_cl.to_csv(OUT / "reg_spicy_weighted_effects_clustered.csv", index=False)
    print("Wrote clustered SE table:", OUT / "reg_spicy_weighted_effects_clustered.csv")
except Exception as e:
    print("Clustered SE export skipped (model_sw_cl not fit or statsmodels missing):", e)

print("Wrote:",
      OUT / "reg_spicy_weighted_preds_by_role_rel_age.csv", "and",
      OUT / "reg_spicy_weighted_effects_hc3.csv")


In [ ]:
# Save the pretty-printed statsmodels summaries as .txt for appendix/attachments
try:
    (OUT / "reg_spicy_weighted_ols_hc3.txt").write_text(str(model_sw.summary()))
    print("Wrote:", OUT / "reg_spicy_weighted_ols_hc3.txt")
except Exception as e:
    print("Could not write HC3 summary:", e)

try:
    (OUT / "reg_spicy_weighted_ols_clustered.txt").write_text(str(model_sw_cl.summary()))
    print("Wrote:", OUT / "reg_spicy_weighted_ols_clustered.txt")
except Exception as e:
    print("Clustered summary not written (optional):", e)
